# OL Report — Optimization & Uncertainty in Learning
## CS7641 Spring 2026

This notebook contains the model, data, and utility code ported from the SL Report.

---
## Models

In [1]:
"""
models.py — PytorchMLP backbone from SL Report with freeze/unfreeze support.
"""
import torch
import torch.nn as nn


class PytorchMLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, output_dim, activation='relu', dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        act_fn = {'relu': nn.ReLU, 'gelu': nn.GELU, 'silu': nn.SiLU, 'tanh': nn.Tanh}
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(act_fn.get(activation, nn.ReLU)())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def count_params(self, only_trainable=True):
        if only_trainable:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

    def freeze_all(self):
        for p in self.parameters():
            p.requires_grad = False

    def unfreeze_last_k_layers(self, k=1):
        """Unfreeze the last k Linear layers (and their biases).
        Keeps everything else frozen. Call freeze_all() first."""
        linear_layers = [m for m in self.net if isinstance(m, nn.Linear)]
        for layer in linear_layers[-k:]:
            for p in layer.parameters():
                p.requires_grad = True

    def get_trainable_params(self):
        return [p for p in self.parameters() if p.requires_grad]

    def get_flat_weights(self):
        """Return trainable parameters as a single flat tensor."""
        params = self.get_trainable_params()
        return torch.cat([p.data.view(-1) for p in params])

    def set_flat_weights(self, flat_tensor):
        """Set trainable parameters from a single flat tensor."""
        offset = 0
        for p in self.get_trainable_params():
            numel = p.numel()
            p.data.copy_(flat_tensor[offset:offset + numel].view(p.shape))
            offset += numel


def build_adult_model(input_dim, dropout=0.2):
    """SL Report backbone for Adult: shallow [200, 200], binary output."""
    return PytorchMLP(input_dim, [200, 200], 1, activation='relu', dropout=dropout)


def build_wine_model(input_dim, n_classes, dropout=0.2):
    """SL Report backbone for Wine: shallow [128, 128], multiclass output."""
    return PytorchMLP(input_dim, [128, 128], n_classes, activation='relu', dropout=dropout)


---
## Data Loading

In [2]:
"""
data.py — Data loading and preprocessing matching the SL Report exactly.
Same splits, same preprocessing, same seed.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

SEED = 42


def load_adult(data_path='data/adult.csv'):
    """Load and preprocess Adult Income dataset.
    Returns: X_train, X_val, X_test, y_train, y_val, y_test (all numpy arrays),
             preprocessor (fitted ColumnTransformer), input_dim.
    """
    adult = pd.read_csv(data_path)
    adult['target'] = (adult['class'].str.strip() == '>50K').astype(int)
    adult = adult.drop(columns=['class'])

    cat_cols = adult.select_dtypes(include=['object']).columns.tolist()
    num_cols = adult.select_dtypes(include=[np.number]).columns.drop('target').tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ]
    )

    X = adult.drop(columns=['target'])
    y = adult['target']

    # Same 80/20 split as SL Report
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # Fit preprocessor on training data only
    X_train_full_proc = preprocessor.fit_transform(X_train_full)
    X_test_proc = preprocessor.transform(X_test)

    # Further split training into train/val (85/15 of training, matching SL Report)
    X_train_proc, X_val_proc, y_train, y_val = train_test_split(
        X_train_full_proc, y_train_full.values, test_size=0.15, random_state=SEED,
        stratify=y_train_full
    )

    y_test = y_test.values
    input_dim = X_train_proc.shape[1]

    return X_train_proc, X_val_proc, X_test_proc, y_train, y_val, y_test, preprocessor, input_dim


def load_wine(data_path='data/wine.csv'):
    """Load and preprocess Wine Quality dataset.
    Returns: X_train, X_val, X_test, y_train, y_val, y_test (all numpy arrays),
             preprocessor (fitted ColumnTransformer), input_dim, n_classes, label_map.
    """
    wine = pd.read_csv(data_path)

    # Remap quality labels to 0-indexed for CrossEntropyLoss
    label_map = {v: k for k, v in enumerate(sorted(wine['quality'].unique()))}
    y = wine['quality'].map(label_map)

    X = wine.drop(columns=['quality', 'class'])
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols)
        ]
    )

    # Same 80/20 split as SL Report
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    X_train_full_proc = preprocessor.fit_transform(X_train_full)
    X_test_proc = preprocessor.transform(X_test)

    # Further split training into train/val (85/15 of training, matching SL Report)
    X_train_proc, X_val_proc, y_train, y_val = train_test_split(
        X_train_full_proc, y_train_full.values, test_size=0.15, random_state=SEED,
        stratify=y_train_full
    )

    y_test = y_test.values
    input_dim = X_train_proc.shape[1]
    n_classes = len(label_map)

    return (X_train_proc, X_val_proc, X_test_proc, y_train, y_val, y_test,
            preprocessor, input_dim, n_classes, label_map)


---
## Utilities

In [3]:
"""
utils.py — Training, evaluation, compute accounting, and seed management.
"""
import time
import copy
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

SEED = 42


def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def to_tensors(X_train, y_train, X_val, y_val, task='binary'):
    """Convert numpy arrays to torch tensors."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X_tr = torch.FloatTensor(X_train).to(device)
    X_v = torch.FloatTensor(X_val).to(device)

    if task == 'binary':
        y_tr = torch.FloatTensor(y_train).to(device)
        y_v = torch.FloatTensor(y_val).to(device)
    else:
        y_tr = torch.LongTensor(y_train).to(device)
        y_v = torch.LongTensor(y_val).to(device)

    return X_tr, y_tr, X_v, y_v, device


def get_criterion(task='binary'):
    if task == 'binary':
        return nn.BCEWithLogitsLoss()
    return nn.CrossEntropyLoss()


def compute_metrics(y_true, y_pred, task='binary'):
    """Compute accuracy and F1 (binary or macro)."""
    acc = accuracy_score(y_true, y_pred)
    if task == 'binary':
        f1 = f1_score(y_true, y_pred)
    else:
        f1 = f1_score(y_true, y_pred, average='macro')
    return acc, f1


def predict(model, X, task='binary'):
    """Predict with a trained model."""
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        out = model(torch.FloatTensor(X).to(device))
        if task == 'binary':
            return (torch.sigmoid(out.squeeze()) > 0.5).cpu().numpy().astype(int)
        else:
            return out.argmax(dim=1).cpu().numpy()


def evaluate_model(model, X, y, task='binary'):
    """Predict and compute metrics."""
    y_pred = predict(model, X, task=task)
    return compute_metrics(y, y_pred, task=task)


def train_model(model, X_train, y_train, X_val, y_val, optimizer, task='binary',
                batch_size=256, max_epochs=100, patience=10, track_grads=True):
    """Train a PyTorch model with the given optimizer.

    Returns: model, history dict, grad_evals count.
    """
    X_tr, y_tr, X_v, y_v, device = to_tensors(X_train, y_train, X_val, y_val, task)
    model = model.to(device)
    criterion = get_criterion(task)

    dataset = torch.utils.data.TensorDataset(X_tr, y_tr)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': [],
        'wall_clock': [],
    }
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    grad_evals = 0
    start_time = time.time()

    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            out = model(xb)
            if task == 'binary':
                loss = criterion(out.squeeze(), yb)
            else:
                loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
            grad_evals += 1
        epoch_loss /= len(X_tr)
        history['train_loss'].append(epoch_loss)

        # Validation
        model.eval()
        with torch.no_grad():
            v_out = model(X_v)
            if task == 'binary':
                v_loss = criterion(v_out.squeeze(), y_v).item()
            else:
                v_loss = criterion(v_out, y_v).item()

        t_acc, t_f1 = evaluate_model(model, X_train, y_train, task)
        v_acc, v_f1 = evaluate_model(model, X_val, y_val, task)

        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        history['train_f1'].append(t_f1)
        history['val_f1'].append(v_f1)
        history['wall_clock'].append(time.time() - start_time)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model, history, grad_evals


def compute_val_loss(model, X_val, y_val, task='binary'):
    """Compute validation loss (one function evaluation for RO accounting)."""
    model.eval()
    device = next(model.parameters()).device
    criterion = get_criterion(task)
    with torch.no_grad():
        X_v = torch.FloatTensor(X_val).to(device)
        if task == 'binary':
            y_v = torch.FloatTensor(y_val).to(device)
            out = model(X_v)
            loss = criterion(out.squeeze(), y_v).item()
        else:
            y_v = torch.LongTensor(y_val).to(device)
            out = model(X_v)
            loss = criterion(out, y_v).item()
    return loss

---
## Baseline Verification
Reproduce SL Report results to confirm everything is wired correctly.

In [4]:
set_seed(42)
X_tr, X_val, X_test, y_tr, y_val, y_test, _, input_dim = load_adult('../data/adult.csv')
print(f'Adult: input_dim={input_dim}, train={X_tr.shape}, val={X_val.shape}, test={X_test.shape}')

model = build_adult_model(input_dim)
print(f'Adult model params: {model.count_params()}')

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-4, momentum=0)
model, hist, grads = train_model(model, X_tr, y_tr, X_val, y_val, optimizer,
                                  task='binary', batch_size=256, max_epochs=100, patience=15)
acc, f1 = evaluate_model(model, X_test, y_test, task='binary')
print(f'Adult Test — Acc: {acc:.4f}, F1: {f1:.4f} (SL baseline: Acc 0.8507, F1 0.6726)')

# Freeze/unfreeze test
model.freeze_all()
model.unfreeze_last_k_layers(k=2)
print(f'Trainable after freeze (last 2 layers): {model.count_params()}')
model.freeze_all()
model.unfreeze_last_k_layers(k=1)
print(f'Trainable after freeze (last 1 layer): {model.count_params()}')

Adult: input_dim=104, train=(30750, 104), val=(5427, 104), test=(9045, 104)
Adult model params: 61401
Adult Test — Acc: 0.8511, F1: 0.6725 (SL baseline: Acc 0.8507, F1 0.6726)
Trainable after freeze (last 2 layers): 40401
Trainable after freeze (last 1 layer): 201


In [5]:
set_seed(42)
(X_tr_w, X_val_w, X_test_w, y_tr_w, y_val_w, y_test_w,
 _, input_dim_w, n_classes, label_map) = load_wine('../data/wine.csv')
print(f'Wine: input_dim={input_dim_w}, n_classes={n_classes}, train={X_tr_w.shape}')

model_w = build_wine_model(input_dim_w, n_classes)
print(f'Wine model params: {model_w.count_params()}')

optimizer_w = torch.optim.SGD(model_w.parameters(), lr=0.01, weight_decay=1e-4, momentum=0)
model_w, hist_w, grads_w = train_model(model_w, X_tr_w, y_tr_w, X_val_w, y_val_w, optimizer_w,
                                        task='multiclass', batch_size=128, max_epochs=100, patience=15)
acc_w, f1_w = evaluate_model(model_w, X_test_w, y_test_w, task='multiclass')
print(f'Wine Test — Acc: {acc_w:.4f}, Macro-F1: {f1_w:.4f} (SL baseline: Acc 0.5638, Macro-F1 0.3085)')

Wine: input_dim=12, n_classes=8, train=(4417, 12)
Wine model params: 19208
Wine Test — Acc: 0.5646, Macro-F1: 0.3198 (SL baseline: Acc 0.5638, Macro-F1 0.3085)


---
## Sanity Check — Shuffled Labels
Train the same backbone on randomly shuffled labels. Performance should collapse to random chance (~50% Adult, ~12.5% Wine), confirming the model learns real patterns, not artifacts.

In [6]:
# Sanity check: shuffled-label baseline
import numpy as np
from torch.optim import SGD

# --- Adult (shuffled) ---
set_seed(42)
X_tr_a, X_val_a, X_test_a, y_tr_a, y_val_a, y_test_a, _, input_dim_a = load_adult("../data/adult.csv")
y_tr_shuffled = y_tr_a.copy()
np.random.seed(99)
np.random.shuffle(y_tr_shuffled)
y_val_shuffled = y_val_a.copy()
np.random.shuffle(y_val_shuffled)

model_sanity_a = PytorchMLP(input_dim_a, [200, 200], 1, "relu", dropout=0.2)
opt_a = SGD(model_sanity_a.parameters(), lr=0.01, weight_decay=1e-4)
X_t, y_t, X_v, y_v, device = to_tensors(X_tr_a, y_tr_shuffled, X_val_a, y_val_shuffled, task="binary")
model_sanity_a, hist_a, _ = train_model(model_sanity_a, X_t, y_t, X_v, y_v,
                                        optimizer=opt_a, task="binary", batch_size=256,
                                        max_epochs=100, patience=15)
preds_a = predict(model_sanity_a, X_test_a, task="binary")
acc_a = accuracy_score(y_test_a, preds_a)
f1_a = f1_score(y_test_a, preds_a)
print(f"Adult (shuffled labels): Acc={acc_a:.4f}, F1={f1_a:.4f}")
print(f"  Expected random chance: ~0.50 Acc, ~0.00 F1")

# --- Wine (shuffled) ---
set_seed(42)
(X_tr_w, X_val_w, X_test_w, y_tr_w, y_val_w, y_test_w,
 _, input_dim_w, n_classes_w, _) = load_wine("../data/wine.csv")
y_tr_w_shuffled = y_tr_w.copy()
np.random.seed(99)
np.random.shuffle(y_tr_w_shuffled)
y_val_w_shuffled = y_val_w.copy()
np.random.shuffle(y_val_w_shuffled)

model_sanity_w = PytorchMLP(input_dim_w, [128, 128], n_classes_w, "relu", dropout=0.2)
opt_w = SGD(model_sanity_w.parameters(), lr=0.01, weight_decay=1e-4)
X_t, y_t, X_v, y_v, device = to_tensors(X_tr_w, y_tr_w_shuffled, X_val_w, y_val_w_shuffled, task="multiclass")
model_sanity_w, hist_w, _ = train_model(model_sanity_w, X_t, y_t, X_v, y_v,
                                        optimizer=opt_w, task="multiclass", batch_size=128,
                                        max_epochs=100, patience=15)
preds_w = predict(model_sanity_w, X_test_w, task="multiclass")
acc_w = accuracy_score(y_test_w, preds_w)
f1_w = f1_score(y_test_w, preds_w, average="macro")
print(f"Wine (shuffled labels): Acc={acc_w:.4f}, Macro-F1={f1_w:.4f}")
print(f"  Expected random chance: ~0.125 Acc, ~0.00 Macro-F1")


Adult (shuffled labels): Acc=0.7521, F1=0.0000
  Expected random chance: ~0.50 Acc, ~0.00 F1
Wine (shuffled labels): Acc=0.3462, Macro-F1=0.0643
  Expected random chance: ~0.125 Acc, ~0.00 Macro-F1


---
## Part 1: Randomized Optimization (Both Datasets)
Apply RHC, SA, GA to the last 2 layers (≤50k trainable params). Objective: validation loss.
Compare to SL baseline on both Adult and Wine.

In [7]:
# ============================================================
# Part 1: Randomized Optimization — Both Datasets
# ============================================================
import copy, time

RO_SEEDS = [42, 123, 456]
MAX_FUNC_EVALS = 2000  # function evaluation budget

def ro_evaluate(model, X_val, y_val, task='binary'):
    """Evaluate model on val set. One function evaluation."""
    return compute_val_loss(model, X_val, y_val, task=task)


def rhc(model, X_val, y_val, task='binary', max_evals=MAX_FUNC_EVALS,
        step_size=0.01, restarts=5, plateau_limit=200):
    """Randomized Hill Climbing with restarts and plateau detection."""
    best_global_loss = float('inf')
    best_global_weights = model.get_flat_weights().clone()
    history = []
    total_evals = 0

    for restart in range(restarts):
        if total_evals >= max_evals:
            break
        # Random restart: perturb from best known
        if restart > 0:
            noise = torch.randn_like(best_global_weights) * 0.1
            model.set_flat_weights(best_global_weights + noise)

        current_loss = ro_evaluate(model, X_val, y_val, task)
        total_evals += 1
        plateau_count = 0

        while total_evals < max_evals and plateau_count < plateau_limit:
            # Perturb weights
            old_weights = model.get_flat_weights().clone()
            perturbation = torch.randn_like(old_weights) * step_size
            model.set_flat_weights(old_weights + perturbation)

            new_loss = ro_evaluate(model, X_val, y_val, task)
            total_evals += 1

            if new_loss < current_loss:
                current_loss = new_loss
                plateau_count = 0
            else:
                model.set_flat_weights(old_weights)
                plateau_count += 1

            if current_loss < best_global_loss:
                best_global_loss = current_loss
                best_global_weights = model.get_flat_weights().clone()
            history.append((total_evals, best_global_loss))

    model.set_flat_weights(best_global_weights)
    return model, history, total_evals


def sa(model, X_val, y_val, task='binary', max_evals=MAX_FUNC_EVALS,
       step_size=0.01, T_init=0.0005, decay=0.995):
    """Simulated Annealing with geometric cooling."""
    current_loss = ro_evaluate(model, X_val, y_val, task)
    best_loss = current_loss
    best_weights = model.get_flat_weights().clone()
    history = []
    T = T_init
    total_evals = 1

    while total_evals < max_evals:
        old_weights = model.get_flat_weights().clone()
        perturbation = torch.randn_like(old_weights) * step_size
        model.set_flat_weights(old_weights + perturbation)

        new_loss = ro_evaluate(model, X_val, y_val, task)
        total_evals += 1
        delta = new_loss - current_loss

        # Accept if better, or probabilistically if worse
        if delta < 0 or np.random.random() < np.exp(-delta / max(T, 1e-10)):
            current_loss = new_loss
        else:
            model.set_flat_weights(old_weights)

        if current_loss < best_loss:
            best_loss = current_loss
            best_weights = model.get_flat_weights().clone()

        T *= decay
        history.append((total_evals, best_loss))

    model.set_flat_weights(best_weights)
    return model, history, total_evals


def ga(model, X_val, y_val, task='binary', max_evals=MAX_FUNC_EVALS,
       pop_size=20, mutation_rate=0.1, mutation_scale=0.01, crossover_rate=0.7,
       elitism=2):
    """Genetic Algorithm with elitism, uniform crossover, Gaussian mutation."""
    base_weights = model.get_flat_weights().clone()
    n_params = base_weights.numel()

    # Initialize population around current weights
    population = [base_weights + torch.randn(n_params) * 0.05 for _ in range(pop_size)]

    # Evaluate initial population
    fitness = []
    total_evals = 0
    for weights in population:
        model.set_flat_weights(weights)
        loss = ro_evaluate(model, X_val, y_val, task)
        fitness.append(loss)
        total_evals += 1

    best_loss = min(fitness)
    best_weights = population[fitness.index(best_loss)].clone()
    history = [(total_evals, best_loss)]

    while total_evals < max_evals:
        # Sort by fitness (lower loss = better)
        paired = sorted(zip(fitness, population), key=lambda x: x[0])
        fitness = [f for f, _ in paired]
        population = [w.clone() for _, w in paired]

        new_pop = []
        # Elitism
        for i in range(min(elitism, len(population))):
            new_pop.append(population[i].clone())

        # Fill rest with crossover + mutation
        while len(new_pop) < pop_size:
            # Tournament selection (k=3)
            idx1 = min(np.random.choice(len(population), 3))
            idx2 = min(np.random.choice(len(population), 3))
            p1, p2 = population[idx1], population[idx2]

            # Uniform crossover
            if np.random.random() < crossover_rate:
                mask = torch.rand(n_params) < 0.5
                child = torch.where(mask, p1, p2)
            else:
                child = p1.clone()

            # Gaussian mutation
            mut_mask = torch.rand(n_params) < mutation_rate
            child += mut_mask.float() * torch.randn(n_params) * mutation_scale
            new_pop.append(child)

        population = new_pop[:pop_size]

        # Evaluate new generation
        fitness = []
        for weights in population:
            if total_evals >= max_evals:
                fitness.append(float('inf'))
                continue
            model.set_flat_weights(weights)
            loss = ro_evaluate(model, X_val, y_val, task)
            fitness.append(loss)
            total_evals += 1

        gen_best = min(fitness)
        if gen_best < best_loss:
            best_loss = gen_best
            best_weights = population[fitness.index(gen_best)].clone()
        history.append((total_evals, best_loss))

    model.set_flat_weights(best_weights)
    return model, history, total_evals


# ============================================================
# Run RO on both datasets
# ============================================================
ro_results = {}  # (dataset, algo) -> list of {history, test_acc, test_f1, func_evals, wall}

for ds_name, load_fn, task, hidden, out_dim, bs in [
    ('Adult', lambda: load_adult('../data/adult.csv'), 'binary', [200, 200], 1, 256),
    ('Wine', lambda: load_wine('../data/wine.csv'), 'multiclass', [128, 128], None, 128)]:

    # Load data
    set_seed(42)
    if ds_name == 'Adult':
        X_tr_d, X_val_d, X_test_d, y_tr_d, y_val_d, y_test_d, _, in_dim = load_fn()
        n_cls = out_dim
    else:
        X_tr_d, X_val_d, X_test_d, y_tr_d, y_val_d, y_test_d, _, in_dim, n_cls, _ = load_fn()

    # First train baseline with SGD (to get starting weights)
    for algo_name, algo_fn in [('RHC', rhc), ('SA', sa), ('GA', ga)]:
        algo_results = []
        for seed in RO_SEEDS:
            set_seed(seed)
            # Build model and train baseline
            if ds_name == 'Adult':
                model_ro = build_adult_model(in_dim)
            else:
                model_ro = PytorchMLP(in_dim, hidden, n_cls, 'relu', dropout=0.2)
            opt_base = torch.optim.SGD(model_ro.parameters(), lr=0.01, weight_decay=1e-4)
            model_ro, _, _ = train_model(model_ro, X_tr_d, y_tr_d, X_val_d, y_val_d,
                                        opt_base, task=task, batch_size=bs,
                                        max_epochs=100, patience=15)
            # Freeze and apply RO to last 2 layers
            model_ro.freeze_all()
            model_ro.unfreeze_last_k_layers(k=2)
            trainable = sum(p.numel() for p in model_ro.parameters() if p.requires_grad)

            start = time.time()
            model_ro, hist_ro, fevals = algo_fn(model_ro, X_val_d, y_val_d, task=task)
            wall = time.time() - start

            acc, f1 = evaluate_model(model_ro, X_test_d, y_test_d, task=task)
            algo_results.append({
                'history': hist_ro, 'test_acc': acc, 'test_f1': f1,
                'func_evals': fevals, 'wall': wall, 'trainable': trainable
            })

        ro_results[(ds_name, algo_name)] = algo_results
        accs = [r['test_acc'] for r in algo_results]
        f1s = [r['test_f1'] for r in algo_results]
        metric_name = 'F1' if task == 'binary' else 'Macro-F1'
        print(f'{ds_name:6s} {algo_name:4s}: Acc={np.mean(accs):.4f}+/-{np.std(accs):.4f}, '
              f'{metric_name}={np.mean(f1s):.4f}+/-{np.std(f1s):.4f}, '
              f'FuncEvals={np.mean([r["func_evals"] for r in algo_results]):.0f}, '
              f'Trainable={algo_results[0]["trainable"]}')

# Print baselines for comparison
print('\nSL Baselines: Adult Acc=0.8507, F1=0.6726 | Wine Acc=0.5638, Macro-F1=0.3085')


Adult  RHC : Acc=0.8498+/-0.0013, F1=0.6732+/-0.0016, FuncEvals=2000, Trainable=40401
Adult  SA  : Acc=0.8482+/-0.0008, F1=0.6677+/-0.0006, FuncEvals=2000, Trainable=40401
Adult  GA  : Acc=0.8509+/-0.0003, F1=0.6758+/-0.0015, FuncEvals=2000, Trainable=40401
Wine   RHC : Acc=0.5500+/-0.0022, Macro-F1=0.3276+/-0.0068, FuncEvals=2000, Trainable=17544
Wine   SA  : Acc=0.5531+/-0.0039, Macro-F1=0.3395+/-0.0113, FuncEvals=2000, Trainable=17544
Wine   GA  : Acc=0.5533+/-0.0016, Macro-F1=0.3230+/-0.0080, FuncEvals=2000, Trainable=17544

SL Baselines: Adult Acc=0.8507, F1=0.6726 | Wine Acc=0.5638, Macro-F1=0.3085


In [8]:
# ============================================================
# Part 1 Plots: RO Progress Curves
# ============================================================
import os
FIGURES_DIR = '../report/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
algo_colors = {'RHC': 'steelblue', 'SA': 'darkorange', 'GA': 'seagreen'}

for di, ds_name in enumerate(['Adult', 'Wine']):
    ax = axes[di]
    for algo_name in ['RHC', 'SA', 'GA']:
        for ri, r in enumerate(ro_results[(ds_name, algo_name)]):
            evals = [h[0] for h in r['history']]
            losses = [h[1] for h in r['history']]
            alpha = 1.0 if ri == 0 else 0.3
            label = algo_name if ri == 0 else None
            ax.plot(evals, losses, color=algo_colors[algo_name],
                    alpha=alpha, label=label, linewidth=1.5 if ri == 0 else 0.8)
    ax.set_xlabel('Function Evaluations')
    ax.set_ylabel('Best Val Loss')
    ax.set_title(f'{ds_name}: RO Progress (Best-So-Far vs Evals)')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/part1_ro_progress.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: part1_ro_progress.png')

# Summary table
print('\nPart 1 RO Summary:')
for ds_name in ['Adult', 'Wine']:
    metric = 'F1' if ds_name == 'Adult' else 'Macro-F1'
    print(f'\n{ds_name}:')
    hdr = f"{'Algo':6s} | {'Test Acc':>9s} | {'Test '+metric:>12s} | {'Func Evals':>10s} | {'Wall (s)':>9s}"
    print(hdr)
    print('-' * 60)
    for algo in ['RHC', 'SA', 'GA']:
        res = ro_results[(ds_name, algo)]
        ta = np.mean([r['test_acc'] for r in res])
        tf = np.mean([r['test_f1'] for r in res])
        fe = np.mean([r['func_evals'] for r in res])
        wl = np.mean([r['wall'] for r in res])
        print(f'{algo:6s} | {ta:>9.4f} | {tf:>12.4f} | {fe:>10.0f} | {wl:>9.2f}')


Saved: part1_ro_progress.png

Part 1 RO Summary:

Adult:
Algo   |  Test Acc |      Test F1 | Func Evals |  Wall (s)
------------------------------------------------------------
RHC    |    0.8498 |       0.6732 |       2000 |      5.42
SA     |    0.8482 |       0.6677 |       2000 |      5.33
GA     |    0.8509 |       0.6758 |       2000 |      5.16

Wine:
Algo   |  Test Acc | Test Macro-F1 | Func Evals |  Wall (s)
------------------------------------------------------------
RHC    |    0.5500 |       0.3276 |       2000 |      0.76
SA     |    0.5531 |       0.3395 |       2000 |      0.78
GA     |    0.5533 |       0.3230 |       2000 |      0.85


---
## Part 2: Adam Ablations (Adult Only)
Train the Adult backbone with 7 optimizer configs. Compare speed-to-threshold, stability across seeds, sensitivity to hyperparameters, and generalization gap.

Optimizers: SGD, SGD+Momentum, Nesterov, Adam, Adam (no bias correction), Adam (β₁=0 / RMSProp-like), AdamW

In [9]:
# ============================================================
# Part 2: Adam Ablations — Adult Only
# ============================================================
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, copy, time
from collections import defaultdict

FIGURES_DIR = "../report/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# --- Custom Adam without bias correction ---
class AdamNoBiasCorrection(optim.Adam):
    """Adam with bias correction disabled (skip the 1/(1-beta^t) step)."""
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p)
                    state["exp_avg_sq"] = torch.zeros_like(p)
                state["step"] += 1
                beta1, beta2 = group["betas"]
                state["exp_avg"].mul_(beta1).add_(grad, alpha=1 - beta1)
                state["exp_avg_sq"].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                # NO bias correction — use raw estimates
                denom = state["exp_avg_sq"].sqrt().add_(group["eps"])
                p.add_(state["exp_avg"] / denom, alpha=-group["lr"])
                if group["weight_decay"] != 0:
                    p.add_(p, alpha=-group["lr"] * group["weight_decay"])


def make_optimizer(name, model, lr=0.001, weight_decay=0):
    """Create one of the 7 required optimizers by name."""
    params = model.parameters()
    configs = {
        "SGD":          lambda: optim.SGD(params, lr=lr),
        "SGD+Momentum": lambda: optim.SGD(params, lr=lr, momentum=0.9),
        "Nesterov":     lambda: optim.SGD(params, lr=lr, momentum=0.9, nesterov=True),
        "Adam":         lambda: optim.Adam(params, lr=lr),
        "Adam-NoBias":  lambda: AdamNoBiasCorrection(params, lr=lr),
        "Adam-B1=0":    lambda: optim.Adam(params, lr=lr, betas=(0.0, 0.999)),
        "AdamW":        lambda: optim.AdamW(params, lr=lr, weight_decay=0.01),
    }
    return configs[name]()


OPTIMIZERS = ["SGD", "SGD+Momentum", "Nesterov", "Adam", "Adam-NoBias", "Adam-B1=0", "AdamW"]
SEEDS = [42, 123, 456, 789, 1024]
LR_DEFAULT = 0.001  # Adam-family default
MAX_EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 256

# --- Load Adult data once ---
set_seed(42)
X_tr, X_val, X_test, y_tr, y_val, y_test, _, input_dim = load_adult("../data/adult.csv")

# ============================================================
# 2A: Train all 7 optimizers across multiple seeds
# ============================================================
results = defaultdict(list)  # opt_name -> list of {history, grad_evals, test_acc, test_f1, wall}

for opt_name in OPTIMIZERS:
    for seed in SEEDS:
        set_seed(seed)
        model = build_adult_model(input_dim)
        # SGD variants need higher LR
        lr = 0.01 if "SGD" in opt_name or opt_name == "Nesterov" else LR_DEFAULT
        optimizer = make_optimizer(opt_name, model, lr=lr)
        model, hist, grads = train_model(
            model, X_tr, y_tr, X_val, y_val, optimizer,
            task="binary", batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS, patience=PATIENCE
        )
        acc, f1 = evaluate_model(model, X_test, y_test, task="binary")
        results[opt_name].append({
            "history": hist, "grad_evals": grads,
            "test_acc": acc, "test_f1": f1,
            "wall": hist["wall_clock"][-1] if hist["wall_clock"] else 0,
            "seed": seed, "lr": lr,
        })
    # Summary for this optimizer
    accs = [r["test_acc"] for r in results[opt_name]]
    f1s = [r["test_f1"] for r in results[opt_name]]
    print(f"{opt_name:15s}: Acc={np.mean(accs):.4f}±{np.std(accs):.4f}, "
          f"F1={np.mean(f1s):.4f}±{np.std(f1s):.4f}, "
          f"AvgGradEvals={np.mean([r['grad_evals'] for r in results[opt_name]]):.0f}")


SGD            : Acc=0.8505±0.0006, F1=0.6742±0.0017, AvgGradEvals=12100
SGD+Momentum   : Acc=0.8523±0.0009, F1=0.6783±0.0029, AvgGradEvals=8349
Nesterov       : Acc=0.8525±0.0006, F1=0.6775±0.0013, AvgGradEvals=8349
Adam           : Acc=0.8514±0.0007, F1=0.6677±0.0082, AvgGradEvals=2565
Adam-NoBias    : Acc=0.8504±0.0011, F1=0.6699±0.0026, AvgGradEvals=2347
Adam-B1=0      : Acc=0.8516±0.0012, F1=0.6740±0.0080, AvgGradEvals=2710
AdamW          : Acc=0.8514±0.0010, F1=0.6678±0.0086, AvgGradEvals=2565


In [10]:
# ============================================================
# 2B: Analysis & Plots
# ============================================================
import matplotlib
matplotlib.use('Agg')

# --- Define fixed validation-loss threshold l ---
all_final_val = []
for opt_name in OPTIMIZERS:
    for r in results[opt_name]:
        all_final_val.append(min(r['history']['val_loss']))
THRESHOLD_L = round(np.percentile(all_final_val, 75), 4)
print(f'Fixed validation-loss threshold l = {THRESHOLD_L}')

# --- Time and steps to threshold l ---
header = f"{'Optimizer':15s} | {'Steps to l':>12s} | {'Time to l (s)':>14s} | {'Final Val Loss':>14s}"
print('\n' + header)
print('-' * 65)
for opt_name in OPTIMIZERS:
    steps_list, time_list = [], []
    for r in results[opt_name]:
        vl = r['history']['val_loss']
        wc = r['history']['wall_clock']
        reached = False
        for i, v in enumerate(vl):
            if v <= THRESHOLD_L:
                steps_list.append(i + 1)
                time_list.append(wc[i])
                reached = True
                break
        if not reached:
            steps_list.append(float('inf'))
            time_list.append(float('inf'))
    finite_steps = [s for s in steps_list if s != float('inf')]
    finite_times = [t for t in time_list if t != float('inf')]
    s_str = f'{np.mean(finite_steps):.1f}+/-{np.std(finite_steps):.1f}' if finite_steps else 'never'
    t_str = f'{np.mean(finite_times):.2f}+/-{np.std(finite_times):.2f}' if finite_times else 'never'
    best_vl = np.mean([min(r['history']['val_loss']) for r in results[opt_name]])
    print(f'{opt_name:15s} | {s_str:>12s} | {t_str:>14s} | {best_vl:>14.4f}')

# ============================================================
# Plot 1: Stability bands -- val loss vs epoch (median + IQR)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(OPTIMIZERS)))

for idx, opt_name in enumerate(OPTIMIZERS):
    max_len = max(len(r['history']['val_loss']) for r in results[opt_name])
    padded = []
    for r in results[opt_name]:
        vl = r['history']['val_loss']
        padded.append(vl + [vl[-1]] * (max_len - len(vl)))
    arr = np.array(padded)
    median = np.median(arr, axis=0)
    q25 = np.percentile(arr, 25, axis=0)
    q75 = np.percentile(arr, 75, axis=0)
    epochs = np.arange(1, max_len + 1)
    axes[0].plot(epochs, median, label=opt_name, color=colors[idx])
    axes[0].fill_between(epochs, q25, q75, alpha=0.2, color=colors[idx])

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Stability Bands: Val Loss vs Epoch (Median + IQR)')
axes[0].legend(fontsize=7, loc='upper right')
axes[0].axhline(y=THRESHOLD_L, color='gray', linestyle='--', alpha=0.5)
axes[0].set_ylim(0.295, 0.36)
axes[0].annotate('SGD starts at ~0.60', xy=(5, 0.358), fontsize=7, fontstyle='italic', color='tab:blue')

# Plot 1b: val loss vs wall clock (with IQR bands)
for idx, opt_name in enumerate(OPTIMIZERS):
    max_len = max(len(r['history']['val_loss']) for r in results[opt_name])
    padded_vl, padded_wc = [], []
    for r in results[opt_name]:
        vl = r['history']['val_loss']
        wc = r['history']['wall_clock']
        padded_vl.append(vl + [vl[-1]] * (max_len - len(vl)))
        padded_wc.append(wc + [wc[-1]] * (max_len - len(wc)))
    arr_vl = np.array(padded_vl)
    arr_wc = np.array(padded_wc)
    median_vl = np.median(arr_vl, axis=0)
    q25_vl = np.percentile(arr_vl, 25, axis=0)
    q75_vl = np.percentile(arr_vl, 75, axis=0)
    median_wc = np.median(arr_wc, axis=0)
    axes[1].plot(median_wc, median_vl, label=opt_name, color=colors[idx])
    axes[1].fill_between(median_wc, q25_vl, q75_vl, alpha=0.2, color=colors[idx])

axes[1].set_xlabel('Wall Clock (s)')
axes[1].set_ylabel('Validation Loss')
axes[1].set_title('Val Loss vs Wall Clock')
axes[1].legend(fontsize=7, loc='upper right')
axes[1].axhline(y=THRESHOLD_L, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylim(0.295, 0.36)
axes[1].annotate('SGD starts at ~0.60', xy=(0.5, 0.358), fontsize=7, fontstyle='italic', color='tab:blue')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/part2_stability_bands.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: part2_stability_bands.png')

# Plot 2: Generalization gap (train vs val vs test acc)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
train_accs, val_accs, test_accs = [], [], []
for opt_name in OPTIMIZERS:
    ta = np.mean([r['history']['train_acc'][-1] for r in results[opt_name]])
    va = np.mean([r['history']['val_acc'][-1] for r in results[opt_name]])
    te = np.mean([r['test_acc'] for r in results[opt_name]])
    train_accs.append(ta)
    val_accs.append(va)
    test_accs.append(te)

x = np.arange(len(OPTIMIZERS))
w = 0.25
ax.bar(x - w, train_accs, w, label='Train Acc', color='steelblue')
ax.bar(x, val_accs, w, label='Val Acc', color='darkorange')
ax.bar(x + w, test_accs, w, label='Test Acc', color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels(OPTIMIZERS, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Accuracy')
ax.set_title('Generalization Gap: Train vs Val vs Test Accuracy')
ax.legend()
ax.set_ylim(0.82, 0.87)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/part2_gen_gap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: part2_gen_gap.png')

# ============================================================
# Summary table
# ============================================================
hdr = f"{'Optimizer':15s} | {'Best Val Loss':>13s} | {'Test Acc':>9s} | {'Test F1':>9s} | {'Grad Evals':>10s} | {'Wall (s)':>9s}"
print('\n' + hdr)
print('-' * 80)
for opt_name in OPTIMIZERS:
    bvl = np.mean([min(r['history']['val_loss']) for r in results[opt_name]])
    ta = np.mean([r['test_acc'] for r in results[opt_name]])
    tf = np.mean([r['test_f1'] for r in results[opt_name]])
    ge = np.mean([r['grad_evals'] for r in results[opt_name]])
    wl = np.mean([r['wall'] for r in results[opt_name]])
    print(f'{opt_name:15s} | {bvl:>13.4f} | {ta:>9.4f} | {tf:>9.4f} | {ge:>10.0f} | {wl:>9.2f}')
print(f'SL Baseline:      Val Loss=N/A,      Acc=0.8507,   F1=0.6726')


Fixed validation-loss threshold l = 0.3106

Optimizer       |   Steps to l |  Time to l (s) | Final Val Loss
-----------------------------------------------------------------
SGD             |        never |          never |         0.3123
SGD+Momentum    |   19.4+/-5.3 |    3.36+/-0.87 |         0.3078
Nesterov        |   18.8+/-5.5 |    3.26+/-1.02 |         0.3078
Adam            |    4.4+/-1.5 |    0.83+/-0.28 |         0.3094
Adam-NoBias     |    3.2+/-1.1 |    0.61+/-0.20 |         0.3099
Adam-B1=0       |    8.5+/-3.5 |    1.67+/-0.59 |         0.3107
AdamW           |    4.4+/-1.5 |    0.84+/-0.28 |         0.3093
Saved: part2_stability_bands.png
Saved: part2_gen_gap.png

Optimizer       | Best Val Loss |  Test Acc |   Test F1 | Grad Evals |  Wall (s)
--------------------------------------------------------------------------------
SGD             |        0.3123 |    0.8505 |    0.6742 |      12100 |     16.89
SGD+Momentum    |        0.3078 |    0.8523 |    0.6783 |       8349

In [11]:
# ============================================================
# 2C: Sensitivity Heatmaps — (alpha, beta1) and (alpha, beta2)
# ============================================================

LR_GRID = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
BETA1_GRID = [0.0, 0.5, 0.8, 0.9, 0.99]
BETA2_GRID = [0.9, 0.99, 0.999, 0.9999]

# --- Heatmap 1: (alpha, beta1) with beta2=0.999 fixed ---
hm_b1 = np.zeros((len(BETA1_GRID), len(LR_GRID)))
print('Running (alpha, beta1) heatmap...')
for i, b1 in enumerate(BETA1_GRID):
    for j, lr in enumerate(LR_GRID):
        set_seed(42)
        model = build_adult_model(input_dim)
        opt = optim.Adam(model.parameters(), lr=lr, betas=(b1, 0.999))
        model, hist, _ = train_model(
            model, X_tr, y_tr, X_val, y_val, opt,
            task='binary', batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS, patience=PATIENCE
        )
        hm_b1[i, j] = min(hist['val_loss'])
        print(f'  b1={b1}, lr={lr}: val_loss={hm_b1[i,j]:.4f}')

# --- Heatmap 2: (alpha, beta2) with beta1=0.9 fixed ---
hm_b2 = np.zeros((len(BETA2_GRID), len(LR_GRID)))
print('Running (alpha, beta2) heatmap...')
for i, b2 in enumerate(BETA2_GRID):
    for j, lr in enumerate(LR_GRID):
        set_seed(42)
        model = build_adult_model(input_dim)
        opt = optim.Adam(model.parameters(), lr=lr, betas=(0.9, b2))
        model, hist, _ = train_model(
            model, X_tr, y_tr, X_val, y_val, opt,
            task='binary', batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS, patience=PATIENCE
        )
        hm_b2[i, j] = min(hist['val_loss'])
        print(f'  b2={b2}, lr={lr}: val_loss={hm_b2[i,j]:.4f}')

# --- Plot heatmaps ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(hm_b1, ax=axes[0], annot=True, fmt='.4f', cmap='YlOrRd_r',
            xticklabels=[f'{lr:.0e}' for lr in LR_GRID],
            yticklabels=[str(b) for b in BETA1_GRID])
axes[0].set_xlabel('Learning Rate (alpha)')
axes[0].set_ylabel('beta1')
axes[0].set_title('Best Val Loss: (alpha, beta1), beta2=0.999')

sns.heatmap(hm_b2, ax=axes[1], annot=True, fmt='.4f', cmap='YlOrRd_r',
            xticklabels=[f'{lr:.0e}' for lr in LR_GRID],
            yticklabels=[str(b) for b in BETA2_GRID])
axes[1].set_xlabel('Learning Rate (alpha)')
axes[1].set_ylabel('beta2')
axes[1].set_title('Best Val Loss: (alpha, beta2), beta1=0.9')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/part2_heatmaps.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: part2_heatmaps.png')


Running (alpha, beta1) heatmap...
  b1=0.0, lr=0.0001: val_loss=0.3083
  b1=0.0, lr=0.0005: val_loss=0.3097
  b1=0.0, lr=0.001: val_loss=0.3108
  b1=0.0, lr=0.005: val_loss=0.3211
  b1=0.0, lr=0.01: val_loss=0.3258
  b1=0.5, lr=0.0001: val_loss=0.3083
  b1=0.5, lr=0.0005: val_loss=0.3090
  b1=0.5, lr=0.001: val_loss=0.3089
  b1=0.5, lr=0.005: val_loss=0.3113
  b1=0.5, lr=0.01: val_loss=0.3174
  b1=0.8, lr=0.0001: val_loss=0.3081
  b1=0.8, lr=0.0005: val_loss=0.3081
  b1=0.8, lr=0.001: val_loss=0.3083
  b1=0.8, lr=0.005: val_loss=0.3115
  b1=0.8, lr=0.01: val_loss=0.3137
  b1=0.9, lr=0.0001: val_loss=0.3082
  b1=0.9, lr=0.0005: val_loss=0.3081
  b1=0.9, lr=0.001: val_loss=0.3086
  b1=0.9, lr=0.005: val_loss=0.3123
  b1=0.9, lr=0.01: val_loss=0.3138
  b1=0.99, lr=0.0001: val_loss=0.3079
  b1=0.99, lr=0.0005: val_loss=0.3077
  b1=0.99, lr=0.001: val_loss=0.3083
  b1=0.99, lr=0.005: val_loss=0.3115
  b1=0.99, lr=0.01: val_loss=0.3120
Running (alpha, beta2) heatmap...
  b2=0.9, lr=0.0001: v

---
## Part 3: Regularization Study (Adult Only)
Standard Adam (lr=0.001, β₁=0.9, β₂=0.999) with best Part 2 hyperparams. Do NOT retune Adam.
Techniques: L2 weight decay (coupled), early stopping, dropout, noise regularization (label smoothing + input noise).

In [12]:
# ============================================================
# Part 3: Regularization Study — Adult Only
# ============================================================
# Adam with best Part 2 hyperparams: lr=0.001, betas=(0.9, 0.999)
# Do NOT retune Adam settings.

PART3_SEEDS = [42, 123, 456, 789, 1024]
PART3_LR = 0.001
PART3_EPOCHS = 100
PART3_PATIENCE = 15

def train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                   weight_decay=0, dropout=0.2, label_smoothing=0.0,
                   input_noise=0.0, patience=PART3_PATIENCE, seeds=PART3_SEEDS):
    """Train Adam with given regularization settings across seeds."""
    all_results = []
    for seed in seeds:
        set_seed(seed)
        model = PytorchMLP(input_dim, [200, 200], 1, 'relu', dropout=dropout)
        opt = optim.Adam(model.parameters(), lr=PART3_LR, weight_decay=weight_decay)

        # Smooth labels for training only
        if label_smoothing > 0:
            y_tr_use = y_tr * (1 - label_smoothing) + 0.5 * label_smoothing
        else:
            y_tr_use = y_tr

        # Add input noise during training only
        if input_noise > 0:
            noise = np.random.randn(*X_tr.shape).astype(np.float32) * input_noise
            X_tr_use = X_tr + noise
        else:
            X_tr_use = X_tr

        # Train with possibly-smoothed labels and noisy inputs
        # But evaluate on original clean data
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)
        criterion = nn.BCEWithLogitsLoss()
        X_t = torch.FloatTensor(X_tr_use).to(device)
        y_t = torch.FloatTensor(y_tr_use).to(device)
        X_v = torch.FloatTensor(X_val).to(device)
        y_v = torch.FloatTensor(y_val).to(device)

        dataset = torch.utils.data.TensorDataset(X_t, y_t)
        loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

        history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
                   'train_f1': [], 'val_f1': [], 'wall_clock': []}
        best_val_loss = float('inf')
        patience_counter = 0
        best_state = None
        grad_evals = 0
        start_time = time.time()

        for epoch in range(PART3_EPOCHS):
            model.train()
            epoch_loss = 0
            for xb, yb in loader:
                opt.zero_grad()
                out = model(xb)
                loss = criterion(out.squeeze(), yb)
                loss.backward()
                opt.step()
                epoch_loss += loss.item() * len(xb)
                grad_evals += 1
            epoch_loss /= len(X_t)
            history['train_loss'].append(epoch_loss)

            model.eval()
            with torch.no_grad():
                v_out = model(X_v)
                v_loss = criterion(v_out.squeeze(), y_v).item()
            # Evaluate on ORIGINAL labels (not smoothed)
            t_acc, t_f1 = evaluate_model(model, X_tr, y_tr, task='binary')
            v_acc, v_f1 = evaluate_model(model, X_val, y_val, task='binary')
            history['val_loss'].append(v_loss)
            history['train_acc'].append(t_acc)
            history['val_acc'].append(v_acc)
            history['train_f1'].append(t_f1)
            history['val_f1'].append(v_f1)
            history['wall_clock'].append(time.time() - start_time)

            if v_loss < best_val_loss:
                best_val_loss = v_loss
                patience_counter = 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break

        if best_state:
            model.load_state_dict(best_state)
        acc, f1 = evaluate_model(model, X_test, y_test, task='binary')
        all_results.append({
            'history': history, 'grad_evals': grad_evals,
            'test_acc': acc, 'test_f1': f1,
            'wall': history['wall_clock'][-1] if history['wall_clock'] else 0,
        })
    return all_results


def summarize(name, res):
    accs = [r['test_acc'] for r in res]
    f1s = [r['test_f1'] for r in res]
    ge = [r['grad_evals'] for r in res]
    wl = [r['wall'] for r in res]
    bvl = [min(r['history']['val_loss']) for r in res]
    print(f'{name:30s}: Acc={np.mean(accs):.4f}+/-{np.std(accs):.4f}, '
          f'F1={np.mean(f1s):.4f}+/-{np.std(f1s):.4f}, '
          f'ValLoss={np.mean(bvl):.4f}, GradEvals={np.mean(ge):.0f}, Wall={np.mean(wl):.2f}s')
    return {'name': name, 'acc': np.mean(accs), 'f1': np.mean(f1s),
            'acc_std': np.std(accs), 'f1_std': np.std(f1s),
            'val_loss': np.mean(bvl), 'grad_evals': np.mean(ge), 'wall': np.mean(wl)}


# --- Baseline: Adam, no added regularization (dropout=0.2 from backbone) ---
print('=== Baseline (Adam, backbone dropout=0.2) ===')
res_baseline = train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                              weight_decay=0, dropout=0.2)
s_baseline = summarize('Baseline (d=0.2)', res_baseline)

# --- No regularization at all ---
print('\n=== No Regularization ===')
res_noreg = train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                           weight_decay=0, dropout=0.0)
s_noreg = summarize('No Reg (d=0.0)', res_noreg)

# --- L2 Weight Decay (coupled) ---
print('\n=== L2 Weight Decay ===')
l2_results = {}
for wd in [1e-5, 1e-4, 1e-3, 1e-2]:
    res = train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                         weight_decay=wd, dropout=0.2)
    l2_results[wd] = summarize(f'L2 wd={wd}', res)

# --- Dropout variations ---
print('\n=== Dropout ===')
do_results = {}
for d in [0.1, 0.3, 0.4, 0.5]:
    res = train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                         dropout=d)
    do_results[d] = summarize(f'Dropout={d}', res)

# --- Early Stopping (vary patience) ---
print('\n=== Early Stopping ===')
es_results = {}
for p in [5, 10, 20, 30]:
    res = train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                         patience=p)
    es_results[p] = summarize(f'Patience={p}', res)

# --- Noise Regularization ---
print('\n=== Noise Regularization ===')
noise_results = {}
for ls, inp_n in [(0.05, 0.0), (0.1, 0.0), (0.0, 0.01), (0.0, 0.05),
                  (0.05, 0.01), (0.1, 0.05)]:
    key = f'LS={ls},IN={inp_n}'
    res = train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                         label_smoothing=ls, input_noise=inp_n)
    noise_results[key] = summarize(key, res)

# --- Best Single Regularizer ---
print('\n=== Finding Best Single Regularizer ===')
all_singles = [s_baseline]
all_singles += list(l2_results.values())
all_singles += list(do_results.values())
all_singles += list(es_results.values())
all_singles += list(noise_results.values())
best_single = max(all_singles, key=lambda x: x['f1'])
print(f'Best single: {best_single["name"]} with F1={best_single["f1"]:.4f}')

# --- Best Combination ---
print('\n=== Best Combination Search ===')
best_wd = max(l2_results.values(), key=lambda x: x['f1'])
best_do = max(do_results.values(), key=lambda x: x['f1'])
best_es = max(es_results.values(), key=lambda x: x['f1'])
best_noise = max(noise_results.values(), key=lambda x: x['f1'])

print(f'Best L2: {best_wd["name"]} (F1={best_wd["f1"]:.4f})')
print(f'Best Dropout: {best_do["name"]} (F1={best_do["f1"]:.4f})')
print(f'Best ES: {best_es["name"]} (F1={best_es["f1"]:.4f})')
print(f'Best Noise: {best_noise["name"]} (F1={best_noise["f1"]:.4f})')

# Extract best hyperparams
best_wd_val = float(best_wd['name'].split('=')[1])
best_do_val = float(best_do['name'].split('=')[1])
best_ls_str = best_noise['name']
parts_n = best_ls_str.split(',')
best_ls_val = float(parts_n[0].split('=')[1])
best_in_val = float(parts_n[1].split('=')[1])

# Combo 1: L2 + best Dropout
res_combo1 = train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                            weight_decay=best_wd_val, dropout=best_do_val)
s_combo1 = summarize(f'L2+Drop (wd={best_wd_val},d={best_do_val})', res_combo1)

# Combo 2: All four
res_combo2 = train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                            weight_decay=best_wd_val, dropout=best_do_val,
                            label_smoothing=best_ls_val, input_noise=best_in_val)
s_combo2 = summarize(f'All4 (wd={best_wd_val},d={best_do_val},ls={best_ls_val},in={best_in_val})', res_combo2)

# Combo 3: Dropout + Noise
res_combo3 = train_with_reg(X_tr, y_tr, X_val, y_val, X_test, y_test, input_dim,
                            dropout=best_do_val,
                            label_smoothing=best_ls_val, input_noise=best_in_val)
s_combo3 = summarize(f'Drop+Noise (d={best_do_val},ls={best_ls_val},in={best_in_val})', res_combo3)

best_combo = max([s_combo1, s_combo2, s_combo3], key=lambda x: x['f1'])
print(f'\nBest combination: {best_combo["name"]} with F1={best_combo["f1"]:.4f}')
print(f'Baseline F1={s_baseline["f1"]:.4f}, SL Baseline F1=0.6726')


=== Baseline (Adam, backbone dropout=0.2) ===
Baseline (d=0.2)              : Acc=0.8514+/-0.0007, F1=0.6677+/-0.0082, ValLoss=0.3094, GradEvals=2565, Wall=4.02s

=== No Regularization ===
No Reg (d=0.0)                : Acc=0.8516+/-0.0014, F1=0.6756+/-0.0098, ValLoss=0.3102, GradEvals=2396, Wall=3.00s

=== L2 Weight Decay ===
L2 wd=1e-05                   : Acc=0.8515+/-0.0005, F1=0.6680+/-0.0081, ValLoss=0.3092, GradEvals=2565, Wall=4.02s
L2 wd=0.0001                  : Acc=0.8514+/-0.0009, F1=0.6727+/-0.0050, ValLoss=0.3089, GradEvals=3049, Wall=4.77s
L2 wd=0.001                   : Acc=0.8520+/-0.0007, F1=0.6776+/-0.0036, ValLoss=0.3065, GradEvals=7454, Wall=11.49s
L2 wd=0.01                    : Acc=0.8473+/-0.0005, F1=0.6577+/-0.0044, ValLoss=0.3202, GradEvals=4477, Wall=7.07s

=== Dropout ===
Dropout=0.1                   : Acc=0.8509+/-0.0010, F1=0.6703+/-0.0055, ValLoss=0.3096, GradEvals=2638, Wall=4.11s
Dropout=0.3                   : Acc=0.8513+/-0.0007, F1=0.6721+/-0.0074,

In [13]:
# ============================================================
# Part 3 Plots
# ============================================================

# Plot: Regularization sweep — F1 comparison
fig, ax = plt.subplots(figsize=(12, 5))

# Collect all results for bar chart
names = ['Baseline\n(d=0.2)', 'No Reg\n(d=0.0)']
f1_means = [s_baseline['f1'], s_noreg['f1']]
f1_stds = [s_baseline['f1_std'], s_noreg['f1_std']]
cat_colors = ['gray', 'lightgray']

# L2
for wd, s in l2_results.items():
    names.append(f'L2\nwd={wd}')
    f1_means.append(s['f1'])
    f1_stds.append(s['f1_std'])
    cat_colors.append('steelblue')

# Dropout
for d, s in do_results.items():
    names.append(f'Drop\n{d}')
    f1_means.append(s['f1'])
    f1_stds.append(s['f1_std'])
    cat_colors.append('darkorange')

# Early stopping
for p, s in es_results.items():
    names.append(f'ES\np={p}')
    f1_means.append(s['f1'])
    f1_stds.append(s['f1_std'])
    cat_colors.append('seagreen')

# Noise (top 3 only to save space)
sorted_noise = sorted(noise_results.items(), key=lambda x: x[1]['f1'], reverse=True)[:3]
for key, s in sorted_noise:
    names.append(f'Noise\n{key}')
    f1_means.append(s['f1'])
    f1_stds.append(s['f1_std'])
    cat_colors.append('mediumpurple')

# Combos
for s in [s_combo1, s_combo2, s_combo3]:
    short = s['name'][:20]
    names.append(f'Combo\n{short}')
    f1_means.append(s['f1'])
    f1_stds.append(s['f1_std'])
    cat_colors.append('crimson')

x = np.arange(len(names))
bars = ax.bar(x, f1_means, yerr=f1_stds, capsize=3, color=cat_colors, alpha=0.85)
ax.axhline(y=0.6726, color='black', linestyle='--', alpha=0.5, label='SL Baseline F1')
ax.axhline(y=s_baseline['f1'], color='gray', linestyle=':', alpha=0.5, label='Part 3 Baseline F1')
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=6, rotation=45, ha='right')
ax.set_ylim(0.64, 0.70)
ax.set_ylabel('Test F1 Score')
ax.set_title('Part 3: Regularization Sweep — Test F1 Under Identical Compute Budget')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/part3_reg_sweep.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: part3_reg_sweep.png')

# Summary table
print('\nPart 3 Summary:')
hdr = f"{'Config':35s} | {'Test Acc':>9s} | {'Test F1':>9s} | {'Best Val Loss':>13s} | {'Grad Evals':>10s}"
print(hdr)
print('-' * 85)
all_p3 = [s_baseline, s_noreg] + list(l2_results.values()) + list(do_results.values())
all_p3 += list(es_results.values()) + list(noise_results.values())
all_p3 += [s_combo1, s_combo2, s_combo3]
for s in all_p3:
    print(f"{s['name']:35s} | {s['acc']:>9.4f} | {s['f1']:>9.4f} | {s['val_loss']:>13.4f} | {s['grad_evals']:>10.0f}")
print(f"{'SL Baseline':35s} | {'0.8507':>9s} | {'0.6726':>9s} | {'---':>13s} | {'---':>10s}")


Saved: part3_reg_sweep.png

Part 3 Summary:
Config                              |  Test Acc |   Test F1 | Best Val Loss | Grad Evals
-------------------------------------------------------------------------------------
Baseline (d=0.2)                    |    0.8514 |    0.6677 |        0.3094 |       2565
No Reg (d=0.0)                      |    0.8516 |    0.6756 |        0.3102 |       2396
L2 wd=1e-05                         |    0.8515 |    0.6680 |        0.3092 |       2565
L2 wd=0.0001                        |    0.8514 |    0.6727 |        0.3089 |       3049
L2 wd=0.001                         |    0.8520 |    0.6776 |        0.3065 |       7454
L2 wd=0.01                          |    0.8473 |    0.6577 |        0.3202 |       4477
Dropout=0.1                         |    0.8509 |    0.6703 |        0.3096 |       2638
Dropout=0.3                         |    0.8513 |    0.6721 |        0.3089 |       2759
Dropout=0.4                         |    0.8510 |    0.6721 |        

In [14]:
# ============================================================
# Part 4: Integrated Best Combination (Extra Credit)
# ============================================================
# Recipe: Adam (best Part 2 HPs) + L2+Dropout (Part 3) + GA fine-tuning (Part 1)
# Constraints: <=200 func evals (10% of Part 1), <=5 interaction trials

import time, copy
EC_SEEDS = [42, 123, 456, 789, 1024]
EC_MAX_EVALS = 200  # 10% of Part 1 budget

ec_results = []

for seed in EC_SEEDS:
    set_seed(seed)
    X_tr, X_val, X_test, y_tr, y_val, y_test, _, in_dim = load_adult('../data/adult.csv')

    # Step 1: Train with best optimizer (Adam) + best regularization (L2+Dropout)
    model = PytorchMLP(in_dim, [200, 200], 1, 'relu', dropout=0.4)  # Part 3 best dropout
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)  # Part 2+3 best
    t0 = time.time()
    model, hist, grad_evals = train_model(
        model, X_tr, y_tr, X_val, y_val, optimizer,
        task='binary', batch_size=256, max_epochs=100, patience=15
    )
    adam_time = time.time() - t0
    adam_val = min(hist['val_loss'])

    # Evaluate before RO
    acc_pre, f1_pre = evaluate_model(model, X_test, y_test, task='binary')

    # Step 2: RO fine-tuning with GA on last 2 layers
    model.freeze_all()
    model.unfreeze_last_k_layers(k=2)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    t1 = time.time()
    model, ro_hist, ro_evals = ga(
        model, X_val, y_val, task='binary',
        max_evals=EC_MAX_EVALS,
        pop_size=10, mutation_rate=0.1, mutation_scale=0.01,
        crossover_rate=0.7, elitism=2
    )
    ro_time = time.time() - t1

    # Unfreeze all for final evaluation
    for p in model.parameters():
        p.requires_grad = True

    acc_post, f1_post = evaluate_model(model, X_test, y_test, task='binary')
    ro_best_val = min(loss for _, loss in ro_hist)

    ec_results.append({
        'seed': seed,
        'adam_val_loss': adam_val,
        'pre_ro_acc': acc_pre, 'pre_ro_f1': f1_pre,
        'post_ro_acc': acc_post, 'post_ro_f1': f1_post,
        'ro_val_loss': ro_best_val,
        'grad_evals': grad_evals, 'func_evals': ro_evals,
        'adam_time': adam_time, 'ro_time': ro_time,
        'trainable': trainable,
    })
    print(f'Seed {seed}: Pre-RO F1={f1_pre:.4f}, Post-RO F1={f1_post:.4f}, '
          f'Delta={f1_post-f1_pre:+.4f}, GradEvals={grad_evals}, FuncEvals={ro_evals}')

# Summary
print('\n=== Part 4 Extra Credit Summary ===')
pre_f1s = [r['pre_ro_f1'] for r in ec_results]
post_f1s = [r['post_ro_f1'] for r in ec_results]
print(f'Pre-RO  F1: {np.median(pre_f1s):.4f} (IQR: {np.percentile(pre_f1s,25):.4f}-{np.percentile(pre_f1s,75):.4f})')
print(f'Post-RO F1: {np.median(post_f1s):.4f} (IQR: {np.percentile(post_f1s,25):.4f}-{np.percentile(post_f1s,75):.4f})')
print(f'Median delta: {np.median([p-q for p,q in zip(post_f1s,pre_f1s)]):+.4f}')
print(f'Total compute: {np.median([r["grad_evals"] for r in ec_results]):.0f} grad + '
      f'{np.median([r["func_evals"] for r in ec_results]):.0f} func evals')
print(f'Total wall time: {np.median([r["adam_time"]+r["ro_time"] for r in ec_results]):.1f}s')
print(f'SL Baseline: F1=0.6726 | Part 3 Best: F1=0.6814')


Seed 42: Pre-RO F1=0.6834, Post-RO F1=0.6800, Delta=-0.0034, GradEvals=6171, FuncEvals=200
Seed 123: Pre-RO F1=0.6789, Post-RO F1=0.6795, Delta=+0.0006, GradEvals=6897, FuncEvals=200
Seed 456: Pre-RO F1=0.6781, Post-RO F1=0.6770, Delta=-0.0011, GradEvals=6413, FuncEvals=200
Seed 789: Pre-RO F1=0.6841, Post-RO F1=0.6766, Delta=-0.0075, GradEvals=7139, FuncEvals=200
Seed 1024: Pre-RO F1=0.6825, Post-RO F1=0.6839, Delta=+0.0014, GradEvals=6655, FuncEvals=200

=== Part 4 Extra Credit Summary ===
Pre-RO  F1: 0.6825 (IQR: 0.6789-0.6834)
Post-RO F1: 0.6795 (IQR: 0.6770-0.6800)
Median delta: -0.0011
Total compute: 6655 grad + 200 func evals
Total wall time: 11.2s
SL Baseline: F1=0.6726 | Part 3 Best: F1=0.6814


In [15]:
# ============================================================
# Missing Figures: Confusion Matrices + Per-Class Analysis
# ============================================================
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib
matplotlib.use('Agg')

# We need to re-train the key models to get predictions for confusion matrices
# 1. SL Baseline (Adult)
# 2. Best RO (Adult: GA, Wine: RHC)
# 3. Best Optimizer (Adult: SGD+Momentum)
# 4. Best Regularization Combo (Adult: L2+Dropout)

# --- Retrain key models for confusion matrices ---
set_seed(42)
X_tr_a, X_val_a, X_test_a, y_tr_a, y_val_a, y_test_a, _, in_dim_a = load_adult('../data/adult.csv')

# SL Baseline
set_seed(42)
m_baseline = build_adult_model(in_dim_a)
opt_bl = torch.optim.SGD(m_baseline.parameters(), lr=0.01, weight_decay=1e-4)
m_baseline, _, _ = train_model(m_baseline, X_tr_a, y_tr_a, X_val_a, y_val_a, opt_bl,
                               task='binary', batch_size=256, max_epochs=100, patience=15)
preds_baseline = predict(m_baseline, X_test_a, task='binary')

# Best Part 2: SGD+Momentum
set_seed(42)
m_sgdm = build_adult_model(in_dim_a)
opt_sgdm = torch.optim.SGD(m_sgdm.parameters(), lr=0.01, momentum=0.9)
m_sgdm, _, _ = train_model(m_sgdm, X_tr_a, y_tr_a, X_val_a, y_val_a, opt_sgdm,
                           task='binary', batch_size=256, max_epochs=100, patience=15)
preds_sgdm = predict(m_sgdm, X_test_a, task='binary')

# Best Part 3: L2+Dropout (wd=0.001, d=0.4)
set_seed(42)
m_reg = PytorchMLP(in_dim_a, [200, 200], 1, 'relu', dropout=0.4)
opt_reg = optim.Adam(m_reg.parameters(), lr=0.001, weight_decay=0.001)
m_reg, _, _ = train_model(m_reg, X_tr_a, y_tr_a, X_val_a, y_val_a, opt_reg,
                          task='binary', batch_size=256, max_epochs=100, patience=15)
preds_reg = predict(m_reg, X_test_a, task='binary')

# Best Part 1 Adult: GA
set_seed(42)
m_ga = build_adult_model(in_dim_a)
opt_ga_base = torch.optim.SGD(m_ga.parameters(), lr=0.01, weight_decay=1e-4)
m_ga, _, _ = train_model(m_ga, X_tr_a, y_tr_a, X_val_a, y_val_a, opt_ga_base,
                         task='binary', batch_size=256, max_epochs=100, patience=15)
m_ga.freeze_all()
m_ga.unfreeze_last_k_layers(k=2)
m_ga, _, _ = ga(m_ga, X_val_a, y_val_a, task='binary')
preds_ga = predict(m_ga, X_test_a, task='binary')

# --- Confusion Matrix Plot (Adult: 2x2 grid) ---
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
labels_a = ['<=50K', '>50K']
configs = [
    ('SL Baseline (SGD)', preds_baseline),
    ('Best Optimizer (SGD+Mom)', preds_sgdm),
    ('Best RO (GA)', preds_ga),
    ('Best Reg (L2+Dropout)', preds_reg),
]
for idx, (name, preds) in enumerate(configs):
    ax = axes[idx // 2][idx % 2]
    cm = confusion_matrix(y_test_a, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels_a, yticklabels=labels_a)
    acc = accuracy_score(y_test_a, preds)
    f1 = f1_score(y_test_a, preds)
    ax.set_title(f'{name}\nAcc={acc:.4f}, F1={f1:.4f}', fontsize=9)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Adult: Confusion Matrices Across Parts', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/confusion_adult.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: confusion_adult.png')

# --- Wine: Best RO (RHC) confusion matrix + per-class F1 ---
set_seed(42)
X_tr_w, X_val_w, X_test_w, y_tr_w, y_val_w, y_test_w, _, in_dim_w, n_cls_w, lmap_w = load_wine('../data/wine.csv')

# Wine SL baseline
set_seed(42)
m_wine_bl = PytorchMLP(in_dim_w, [128, 128], n_cls_w, 'relu', dropout=0.2)
opt_wbl = torch.optim.SGD(m_wine_bl.parameters(), lr=0.01, weight_decay=1e-4)
m_wine_bl, _, _ = train_model(m_wine_bl, X_tr_w, y_tr_w, X_val_w, y_val_w, opt_wbl,
                              task='multiclass', batch_size=128, max_epochs=100, patience=15)
preds_wine_bl = predict(m_wine_bl, X_test_w, task='multiclass')

# Wine RHC
set_seed(42)
m_wine_rhc = PytorchMLP(in_dim_w, [128, 128], n_cls_w, 'relu', dropout=0.2)
opt_wrhc = torch.optim.SGD(m_wine_rhc.parameters(), lr=0.01, weight_decay=1e-4)
m_wine_rhc, _, _ = train_model(m_wine_rhc, X_tr_w, y_tr_w, X_val_w, y_val_w, opt_wrhc,
                               task='multiclass', batch_size=128, max_epochs=100, patience=15)
m_wine_rhc.freeze_all()
m_wine_rhc.unfreeze_last_k_layers(k=2)
m_wine_rhc, _, _ = rhc(m_wine_rhc, X_val_w, y_val_w, task='multiclass')
preds_wine_rhc = predict(m_wine_rhc, X_test_w, task='multiclass')

# Wine confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Get class label names
inv_lmap = {v: k for k, v in lmap_w.items()} if lmap_w else {i: i for i in range(n_cls_w)}
wine_labels = [str(inv_lmap.get(i, i)) for i in range(n_cls_w)]

for idx, (name, preds) in enumerate([('SL Baseline', preds_wine_bl), ('Best RO (RHC)', preds_wine_rhc)]):
    cm = confusion_matrix(y_test_w, preds, labels=list(range(n_cls_w)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[idx],
                xticklabels=wine_labels, yticklabels=wine_labels)
    acc = accuracy_score(y_test_w, preds)
    mf1 = f1_score(y_test_w, preds, average='macro')
    axes[idx].set_title(f'Wine: {name}\nAcc={acc:.4f}, Macro-F1={mf1:.4f}', fontsize=10)
    axes[idx].set_ylabel('Actual Quality')
    axes[idx].set_xlabel('Predicted Quality')

plt.suptitle('Wine: Confusion Matrices — SL Baseline vs Best RO', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/confusion_wine.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: confusion_wine.png')

# --- Per-class F1 breakdown for Wine ---
fig, ax = plt.subplots(figsize=(10, 5))
from sklearn.metrics import f1_score as f1_per_class
f1_bl_per = f1_per_class(y_test_w, preds_wine_bl, average=None, labels=list(range(n_cls_w)))
f1_rhc_per = f1_per_class(y_test_w, preds_wine_rhc, average=None, labels=list(range(n_cls_w)))

x = np.arange(n_cls_w)
w = 0.35
ax.bar(x - w/2, f1_bl_per, w, label='SL Baseline', color='steelblue')
ax.bar(x + w/2, f1_rhc_per, w, label='Best RO (RHC)', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels([f'Quality {wine_labels[i]}' for i in range(n_cls_w)], rotation=30, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Wine: Per-Class F1 — SL Baseline vs RHC')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/wine_perclass_f1.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: wine_perclass_f1.png')

# Print per-class report
print('\nWine Per-Class F1:')
for i in range(n_cls_w):
    print(f'  Quality {wine_labels[i]}: Baseline={f1_bl_per[i]:.4f}, RHC={f1_rhc_per[i]:.4f}, '
          f'Delta={f1_rhc_per[i]-f1_bl_per[i]:+.4f}')


Saved: confusion_adult.png
Saved: confusion_wine.png
Saved: wine_perclass_f1.png

Wine Per-Class F1:
  Quality 1: Baseline=0.0000, RHC=0.0000, Delta=+0.0000
  Quality 2: Baseline=0.0000, RHC=0.0000, Delta=+0.0000
  Quality 3: Baseline=0.5862, RHC=0.5753, Delta=-0.0109
  Quality 4: Baseline=0.6183, RHC=0.5928, Delta=-0.0255
  Quality 5: Baseline=0.5436, RHC=0.5186, Delta=-0.0250
  Quality 6: Baseline=0.5000, RHC=0.5149, Delta=+0.0149
  Quality 7: Baseline=0.3103, RHC=0.4384, Delta=+0.1280
  Quality 8: Baseline=0.0000, RHC=0.0000, Delta=+0.0000
